# 📘 Session 16: Data Preprocessing for Machine Learning — Foundations
### Duration: ~2.5–3 Hours

---

**Topics Covered:**
0. What Is Machine Learning? (Concepts Only)
1. From EDA Table to ML Table — Features and Target
2. Train / Test Split
3. Handling Missing Values
4. Encoding Categorical Variables
5. Scaling Numeric Features
6. ColumnTransformer and Pipeline
7. Feature Choices and Preprocessing Documentation

---

**Why This Session Matters for Data Science:**
- **Models cannot read messy tables** — they need consistent numeric inputs
- **Session 15 EDA** told us *what* to fix; preprocessing is *how* we fix it safely
- **Wrong preprocessing causes leakage** — models look great in notebooks but fail in production
- **Pipelines** make preprocessing repeatable and testable

> **Prerequisite mindset:** You completed EDA on penguins in Session 15. We continue that dataset here. **No prior ML experience required.**

> **Session 17 preview:** We will train our **first models** on the preprocessed data you build today.

**Libraries:** `pandas`, `numpy`, `seaborn`, **`scikit-learn`** (`pip install scikit-learn` if needed)


---
## 0. What Is Machine Learning? (Concepts Only)

**Machine learning (ML)** means learning patterns from data so the computer can **predict** or **classify** on new, unseen examples.

### Supervised vs unsupervised (first pass)

| Type | You have labels? | Example |
|------|------------------|---------|
| **Supervised** | Yes | Predict penguin **species** from measurements |
| **Unsupervised** | No | Group similar penguins without species column |

This course starts with **supervised** learning.

### Features vs target

| Symbol | Meaning | Penguin example |
|--------|---------|-----------------|
| **X** | Input features (predictors) | bill length, body mass, island, sex |
| **y** | Target (what we predict) | `species` |

### Why EDA came first (Session 15)

| EDA finding | Preprocessing action (this session) |
|-------------|-------------------------------------|
| Missing values | Impute or drop rows |
| Categorical columns | One-hot encode |
| Different numeric scales | Standardize / scale |
| Species imbalance | Stratified train/test split |
| Correlated features | Document; feature selection later |

### What is preprocessing?

**Preprocessing** = all steps that turn a raw DataFrame into a **model-ready numeric matrix** while avoiding **data leakage**.

> ⚠️ **Special Case — leakage**: If you compute the mean of `body_mass_g` using the **full dataset** before splitting, test-row information leaked into training. **Split first**, then fit transformers on **train only**.

> **Data Science relevance**: Most beginner ML failures are preprocessing mistakes, not wrong algorithms.


---
## 1. From EDA Table to ML Table

We continue the Palmer Penguins project from Session 15.

**Modeling question:** Can we predict **`species`** from measurements and metadata?

| Role | Columns |
|------|---------|
| **Target (y)** | `species` |
| **Features (X)** | numeric measurements + `island`, `sex` (+ optional `bill_ratio` from Session 15) |

We will **not** put `species` inside `X`.


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns

raw = sns.load_dataset("penguins")
print("Raw shape:", raw.shape)
print("\nMissing %:\n", (raw.isna().mean() * 100).round(2))

# Session 15 style complete-case table (for comparison later)
complete = raw.dropna(
    subset=[
        "species", "island", "sex",
        "bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g",
    ]
).copy()
complete["bill_ratio"] = complete["bill_length_mm"] / complete["bill_depth_mm"]

print(f"\nComplete-case rows: {len(complete)} (dropped {len(raw) - len(complete)})")

# Define X and y
y = complete["species"]
X = complete.drop(columns=["species"])

print("\nX shape:", X.shape)
print("y shape:", y.shape)
print("\nX dtypes:\n", X.dtypes)
print("\nTarget counts:\n", y.value_counts())


---
## 2. Train / Test Split — Why and How

We hold out a **test set** to simulate performance on **unseen** penguins.

| Parameter | Purpose |
|-----------|---------|
| `test_size=0.2` | 20% for testing, 80% for training |
| `stratify=y` | Keep species proportions similar in train and test |
| `random_state=42` | Reproducible split |

> ⚠️ **Special Case — order of operations**: **Split → fit preprocessors on train → transform train and test.** Never fit on the full dataset.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("\nSpecies % in full data:\n", y.value_counts(normalize=True).round(3))
print("\nSpecies % in train:\n", y_train.value_counts(normalize=True).round(3))
print("\nSpecies % in test:\n", y_test.value_counts(normalize=True).round(3))


---
## 3. Handling Missing Values

Session 15 used **complete-case** analysis (drop rows with any missing value). That is valid when missingness is small.

| Strategy | When | Tool |
|----------|------|------|
| Drop rows | Few missing, acceptable data loss | `dropna()` |
| Impute numeric | Keep rows; fill with median/mean | `SimpleImputer` |
| Impute categorical | Fill with most frequent category | `SimpleImputer(strategy='most_frequent')` |

We compare row counts on **raw** data: complete-case vs imputed (without splitting yet, for illustration only).


In [ ]:
from sklearn.impute import SimpleImputer

# Track A: complete-case (already have `complete`)
print("Track A complete-case rows:", len(complete))

# Track B: impute on raw (illustration — in production, imputer fits on TRAIN only)
raw_imp = raw.copy()
num_cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
cat_cols = ["island", "sex"]

num_imputer = SimpleImputer(strategy="median")
raw_imp[num_cols] = num_imputer.fit_transform(raw_imp[num_cols])

cat_imputer = SimpleImputer(strategy="most_frequent")
raw_imp[cat_cols] = cat_imputer.fit_transform(raw_imp[cat_cols])

raw_imp = raw_imp.dropna(subset=["species"])
print("Track B after imputation (species present):", len(raw_imp))

print("\nLesson: imputation keeps more rows, but you must fit imputers on training data only in real pipelines.")


---
## 4. Encoding Categorical Variables

Most ML models require **numbers**, not strings like `'Male'` or `'Biscoe'`.

| Method | Use case |
|--------|----------|
| **OneHotEncoder** | Nominal categories (`island`, `sex`) — no natural order |
| **OrdinalEncoder** | Ordered categories only (e.g., low/med/high) |
| **`pd.get_dummies`** | Quick pandas version — harder to reuse in production pipelines |

We demonstrate on **training data only** for `island`.


In [ ]:
from sklearn.preprocessing import OneHotEncoder

# pandas approach (quick demo)
dummies = pd.get_dummies(X_train["island"], prefix="island")
print("pd.get_dummies columns:", list(dummies.columns))
print(dummies.head())

# sklearn approach (pipeline-friendly)
enc = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
island_train = enc.fit_transform(X_train[["island"]])
print("\nOneHotEncoder feature names:", enc.get_feature_names_out(["island"]))
print("Encoded shape:", island_train.shape)


---
## 5. Scaling Numeric Features

Numeric columns can have very different ranges (bill mm vs body mass in grams). Many models behave better when features share a similar scale.

| Scaler | Effect | Typical use |
|--------|--------|-------------|
| **StandardScaler** | Mean 0, std 1 | Default for many linear models |
| **MinMaxScaler** | Scale to [0, 1] | When bounded range is needed |

> ⚠️ **Special Case — fit vs transform**: `fit` learns mean/std from **train**; `transform` applies to train **and** test. Never `fit` on test.

> **Preview for Session 17:** Tree models (Random Forest) often work **without** scaling — we still pipeline them for consistency.


In [ ]:
from sklearn.preprocessing import StandardScaler

num_features = [
    "bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g", "bill_ratio",
]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train[num_features])
X_test_scaled = scaler.transform(X_test[num_features])

print("Train body_mass_g mean (raw):", X_train["body_mass_g"].mean().round(2))
print("Train body_mass_g mean (scaled):", X_train_scaled[:, 3].mean().round(4))
print("Test body_mass_g mean (scaled) — should NOT be ~0:", X_test_scaled[:, 3].mean().round(4))


---
## 6. ColumnTransformer and Pipeline — Putting It All Together

This is the **main pattern** for real projects:

```
train_test_split
    → Pipeline(
          ColumnTransformer(
              numeric: Imputer + Scaler,
              categorical: Imputer + OneHotEncoder
          )
      )
    → model-ready X_train, X_test
```

We build the full pipeline on penguins and produce processed matrices for Session 17.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

numeric_features = [
    "bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g", "bill_ratio",
]
categorical_features = ["island", "sex"]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features),
])

# Full preprocessing pipeline (model step added in Session 17)
prep_pipeline = Pipeline([
    ("preprocessor", preprocessor),
])

X_train_processed = prep_pipeline.fit_transform(X_train, y_train)
X_test_processed = prep_pipeline.transform(X_test)

feature_names = prep_pipeline.named_steps["preprocessor"].get_feature_names_out()
print("Processed train shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)
print("\nFirst 8 feature names:", feature_names[:8])
print("Total features:", len(feature_names))

# Session 17 teaser — not explained yet
# from sklearn.linear_model import LogisticRegression
# clf = LogisticRegression(max_iter=1000)
# clf.fit(X_train_processed, y_train)


---
## 7. Feature Choices and Documentation

| Decision | Our choice | Rationale |
|----------|------------|-----------|
| Target | `species` | Session 15 classification hypothesis |
| Keep `bill_ratio` | Yes | Engineered in Session 15 |
| Drop `flipper_length_mm` now? | No | High correlation with mass — note for later feature selection |
| Island in features | Yes | Encode with one-hot; interpret carefully (confounding) |
| Split | 80/20 stratified | Preserve species mix |

Write a short preprocessing memo whenever you hand data to modeling.


In [ ]:
from pathlib import Path

memo_lines = [
    "# Preprocessing Memo — Palmer Penguins",
    "",
    "## Modeling goal",
    "- Predict species from measurements + island + sex",
    "",
    "## Split",
    f"- train_test_split: 80/20, stratify=species, random_state=42",
    f"- Train rows: {len(X_train)} | Test rows: {len(X_test)}",
    "",
    "## Missing values",
    "- Complete-case subset before split (Session 15 approach)",
    "- Pipeline imputers ready if we switch to imputation strategy later",
    "",
    "## Encoding",
    "- OneHotEncoder on island, sex (handle_unknown=ignore)",
    "",
    "## Scaling",
    "- StandardScaler on numeric features (median impute first)",
    "",
    "## Output",
    f"- Processed feature count: {X_train_processed.shape[1]}",
    "- Artifacts: model-ready X_train_processed, X_test_processed, y_train, y_test",
    "",
    "## Next (Session 17)",
    "- Logistic Regression and Random Forest on processed data",
    "- Accuracy and confusion matrix",
]

memo = "\n".join(memo_lines)
print(memo)
Path("penguin_preprocessing_report.md").write_text(memo, encoding="utf-8")
print("\nWrote penguin_preprocessing_report.md")


---
## 🧪 Practice Exercises

1. Define `X` and `y` for predicting `species` from raw penguins (after your chosen cleaning). Print shapes and dtypes.
2. Split 75/25 with `stratify=y`. Show species proportions in train vs test — are they close?
3. Compare row counts: complete-case vs imputed raw data (Track A vs Track B above).
4. One-hot encode `island` with `pd.get_dummies`, then repeat with `OneHotEncoder` on train only.
5. Scale `body_mass_g` with `StandardScaler`: train mean ≈ 0 after scaling; test mean should **not** be refit to 0.
6. List which columns you would put in **numeric** vs **categorical** branches of `ColumnTransformer`.
7. Fit the full `prep_pipeline` on `X_train`; transform `X_test`. Report final feature count.
8. Write a 6-bullet preprocessing memo for a **different** target: predict `body_mass_g` instead of `species`. What changes in `y` and `X`?

---

## 📝 Session 16 Summary

| Topic | Key takeaway |
|-------|--------------|
| **ML basics** | Supervised learning uses **X** to predict **y** |
| **X / y split** | Never include the target in features |
| **Train / test** | Split first; use `stratify` for imbalanced targets |
| **Missing data** | Drop or impute — document the rule |
| **Encoding** | Categories → one-hot numeric columns |
| **Scaling** | Fit scaler on train; transform test only |
| **Pipeline** | `ColumnTransformer` + `Pipeline` = reproducible preprocessing |

### Key gotchas
- Fitting scaler/imputer on **full data before split** → leakage
- Calling **`fit_transform` on test data** → leakage
- Leaving **`species` in X** when predicting species → cheating
- One-hot encoding **high-cardinality text** without thinking → too many columns

### Next Session
**Session 17**: Introduction to Supervised Learning — first models (Logistic Regression and Random Forest), `fit` / `predict`, accuracy, and confusion matrix on the **preprocessed penguins data** from this session.
